In [ ]:
import random
def route_distance(route, distances):
    total_distance = 0
    for i in range(len(route) - 1):  # Iterate through city pairs
        city1, city2 = route[i], route[i + 1]
        distance = distances.get((city1, city2), distances.get((city2, city1), float('inf')))
        if distance == float('inf'):
            print(f"Warning: No distance found between {city1} and {city2}")
        total_distance += distance
    return total_distance

# Fitness function: inversely proportional to route distance
def fitness(route, distances):
    total_distance = route_distance(route, distances)
    return 1 / total_distance if total_distance > 0 else float('inf')

# Initialize a population of random routes
def initialize_population(cities, population_size, start_city=None):
    population = []
    for _ in range(population_size):
        route = cities[:]
        if start_city:
            route.remove(start_city)  # Remove the starting city
            random.shuffle(route)
            route = [start_city] + route + [start_city]  # Add start city at beginning and end
        else:
            random.shuffle(route)
            route.append(route[0])  # Close the loop for non-fixed start
        population.append(route)
    return population

# Roulette-wheel selection based on fitness
def selection(population, distances):
    fitness_scores = [fitness(route, distances) for route in population]
    total_fitness = sum(fitness_scores)
    if total_fitness == 0:  # Prevent division by zero
        return random.sample(population, 2)
    probabilities = [score / total_fitness for score in fitness_scores]
    return random.choices(population, weights=probabilities, k=2)

def crossover(parent1, parent2):
    # Select a random crossover point
    crossover_point = random.randint(1, len(parent1) - 2)  # Exclude the start and end cities
    # Take the 2 segments from both parents
    first_segment = parent1[:crossover_point]
    second_segment = parent2[crossover_point:-1]  # Exclude the final city of parent2

    child = first_segment + second_segment
    # Validate for duplicates and missing cities
    if len(set(child)) == len(parent1) - 1:  # Ensure all unique cities are present
        child.append(child[0])  # Close the loop by appending the start city
        return child
    else:
        # Return None if the child is invalid
        return None

# Mutation by swapping two cities in the middle of the route
def mutate(route, mutation_rate):
    if random.random() < mutation_rate:
        i, j = random.sample(range(1, len(route) - 1), 2)  # Exclude start and end cities
        route[i], route[j] = route[j], route[i]
    return route

# Generate the next generation of the population
def next_generation(population, distances, mutation_rate):
    new_population = []
    for _ in range(len(population)):
        parent1, parent2 = select_parents(population, distances)
        child = crossover(parent1, parent2)
        child = mutate(child, mutation_rate)
        new_population.append(child)
    return new_population

# Genetic Algorithm
def genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations, start_city=None, verbose=True):
    # Validate the starting city if provided
    if start_city and start_city not in cities:
        raise ValueError(f"Starting city '{start_city}' is not in the list of cities: {cities}")
    population = initialize_population(cities, population_size, start_city)
    for generation in range(1, max_generations + 1):
        if verbose:
            print(f"Generation: {generation}")
        # Calculate fitness for the current population
        fitness_population = [(route, fitness(route, distances)) for route in population]
        fitness_population = sorted(fitness_population, key=lambda x: x[1], reverse=True)
        # Preserve the top 10% (elite individuals)
        elite_count = max(1, population_size // 10)
        elites = [route for route, _ in fitness_population[:elite_count]]
        # Print the best route and its fitness if verbose is True
        best_route, best_fitness = fitness_population[0]
        if verbose:
            print(f"Best Route: {' -> '.join(best_route)} Distance: {1 / best_fitness:.2f}")
        selected_population = selection([route for route, _ in fitness_population], distances)
        # Generate new population
        new_population = elites[:]  # add elites
        max_attempts = 100  # To prevent infinite loops
        attempts = 0
        while len(new_population) < population_size and attempts < max_attempts:
            parent1, parent2 = random.sample(selected_population, 2)
            child = crossover(parent1, parent2)
            if child:  # Only accept valid children
                child = mutate(child, mutation_rate)
                new_population.append(child)
            attempts += 1
        # If max_attempts is reached
        if len(new_population) < population_size:
            if verbose:
                print(f"Warning: max_attempts reached. Population not fully regenerated. "
                      f"Created {len(new_population)} individuals out of {population_size}.")
            missing_count = population_size - len(new_population)
            if verbose:
                print(f"Filling {missing_count} missing individuals with random routes.")
            for _ in range(missing_count):
                random_route = cities[:]
                random.shuffle(random_route)
                if start_city:
                    random_route = [start_city] + [city for city in random_route if city != start_city] + [start_city]
                else:
                    random_route.append(random_route[0])  # Close the loop
                new_population.append(random_route)
        # Replace the old population with the new generation
        population = new_population
    best_route, best_fitness = sorted([(route, fitness(route, distances)) for route in population], key=lambda x: x[1], reverse=True)[0]
    return best_route, 1 / best_fitness

In [ ]:
# TravOrder Data
distances = {
    ('B', 'L'): 3,
    ('L', 'M'): 4.5,
    ('M', 'S'): 3.1,
    ('S', 'L'): 5.7,
    ('B', 'M'): 7.6,
    ('B', 'S'): 7.8
}

      # Boston, London, Mumbai, Shanghai
cities = ['B', 'L', 'M', 'S']

population_size = 100
mutation_rate = 0.2
max_generations = 10

best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'B')

print("\nBest Route Found:", " -> ".join(best_route))
print(f"Best Distance: {best_distance:.2f}")

Generation: 1
Best Route: B -> L -> M -> S -> B Distance: 18.40
Generation: 2
Best Route: B -> L -> M -> S -> B Distance: 18.40
Filling 60 missing individuals with random routes.
Generation: 3
Best Route: B -> L -> M -> S -> B Distance: 18.40
Filling 21 missing individuals with random routes.
Generation: 4
Best Route: B -> L -> M -> S -> B Distance: 18.40
Generation: 5
Best Route: B -> L -> M -> S -> B Distance: 18.40
Generation: 6
Best Route: B -> L -> M -> S -> B Distance: 18.40
Generation: 7
Best Route: B -> L -> M -> S -> B Distance: 18.40
Generation: 8
Best Route: B -> L -> M -> S -> B Distance: 18.40
Filling 29 missing individuals with random routes.
Generation: 9
Best Route: B -> L -> M -> S -> B Distance: 18.40
Filling 27 missing individuals with random routes.
Generation: 10
Best Route: B -> L -> M -> S -> B Distance: 18.40
Filling 22 missing individuals with random routes.

Best Route Found: B -> L -> M -> S -> B
Best Distance: 18.40


In [ ]:
distances = {
    ('B', 'L'): 3,
    ('L', 'M'): 4.5,
    ('M', 'S'): 3.1,
    ('S', 'L'): 5.7,
    ('B', 'M'): 7.6,
    ('B', 'S'): 7.8
}

      # Boston, London, Mumbai, Shanghai
cities = ['B', 'L', 'M', 'S']

population_size = 100
mutation_rate = 0.2
max_generations = 10

for i in range(10):
  best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'B', verbose= False)
  print(f'Run {i+1}')
  print("Best Route Found:", " -> ".join(best_route))
  print(f"Best Distance: {best_distance:.2f}\n")

Run 1
Best Route Found: B -> S -> M -> L -> B
Best Distance: 18.40

Run 2
Best Route Found: B -> L -> M -> S -> B
Best Distance: 18.40

Run 3
Best Route Found: B -> S -> M -> L -> B
Best Distance: 18.40

Run 4
Best Route Found: B -> S -> M -> L -> B
Best Distance: 18.40

Run 5
Best Route Found: B -> L -> M -> S -> B
Best Distance: 18.40

Run 6
Best Route Found: B -> S -> M -> L -> B
Best Distance: 18.40

Run 7
Best Route Found: B -> S -> M -> L -> B
Best Distance: 18.40

Run 8
Best Route Found: B -> L -> M -> S -> B
Best Distance: 18.40

Run 9
Best Route Found: B -> S -> M -> L -> B
Best Distance: 18.40

Run 10
Best Route Found: B -> S -> M -> L -> B
Best Distance: 18.40



In [ ]:
# Illusrative Data

cities = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
# Generate random distances between the cities
def generate_pairwise_distances(cities, low=5, high=100, median=20, seed=None):
    if seed:
        random.seed(seed)  # For reproducibility
    distances = {}
    for i, city1 in enumerate(cities):
        for j, city2 in enumerate(cities):
            if i < j:  # Avoid duplicates and self-distances
                # Generate random distance using a normal distribution around the median
                random_distance = random.gauss(median, median / 2)  # Standard deviation = median / 2
                random_distance = max(low, min(random_distance, high))  # Clamp between low and high
                distances[(city1, city2)] = round(random_distance, 2)
    return distances

distances = generate_pairwise_distances(cities, low=5, high=100, seed=42)

distances

{('A', 'B'): 18.56,
 ('A', 'C'): 18.27,
 ('A', 'D'): 18.89,
 ('A', 'E'): 27.02,
 ('A', 'F'): 18.72,
 ('A', 'G'): 5.03,
 ('A', 'H'): 23.32,
 ('A', 'I'): 17.33,
 ('A', 'J'): 17.83,
 ('B', 'C'): 21.16,
 ('B', 'D'): 22.32,
 ('B', 'E'): 31.64,
 ('B', 'F'): 26.57,
 ('B', 'G'): 21.11,
 ('B', 'H'): 12.62,
 ('B', 'I'): 9.85,
 ('B', 'J'): 22.46,
 ('C', 'D'): 33.11,
 ('C', 'E'): 20.42,
 ('C', 'F'): 18.94,
 ('C', 'G'): 25.32,
 ('C', 'H'): 5.46,
 ('C', 'I'): 16.88,
 ('C', 'J'): 24.9,
 ('D', 'E'): 28.73,
 ('D', 'F'): 17.59,
 ('D', 'G'): 23.77,
 ('D', 'H'): 22.48,
 ('D', 'I'): 27.82,
 ('D', 'J'): 8.87,
 ('E', 'F'): 25.68,
 ('E', 'G'): 5,
 ('E', 'H'): 5,
 ('E', 'I'): 13.93,
 ('E', 'J'): 10.84,
 ('F', 'G'): 28.76,
 ('F', 'H'): 26.64,
 ('F', 'I'): 7.81,
 ('F', 'J'): 28.47,
 ('G', 'H'): 9.98,
 ('G', 'I'): 19.14,
 ('G', 'J'): 17.06,
 ('H', 'I'): 21.14,
 ('H', 'J'): 28.19,
 ('I', 'J'): 26.38}

In [ ]:
population_size = 100
mutation_rate = 0.2
max_generations = 10

best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'A')

print("\nBest Route Found:", " -> ".join(best_route))
print(f"Best Distance: {best_distance:.2f}")

Generation: 1
Best Route: A -> G -> J -> E -> H -> C -> B -> D -> I -> F -> A Distance: 141.22
Filling 78 missing individuals with random routes.
Generation: 2
Best Route: A -> G -> J -> E -> H -> C -> B -> D -> I -> F -> A Distance: 141.22
Filling 80 missing individuals with random routes.
Generation: 3
Best Route: A -> G -> J -> E -> H -> C -> B -> D -> I -> F -> A Distance: 141.22
Filling 80 missing individuals with random routes.
Generation: 4
Best Route: A -> G -> B -> I -> F -> C -> H -> D -> J -> E -> A Distance: 137.41
Filling 77 missing individuals with random routes.
Generation: 5
Best Route: A -> G -> B -> I -> F -> C -> H -> D -> J -> E -> A Distance: 137.41
Filling 67 missing individuals with random routes.
Generation: 6
Best Route: A -> G -> E -> H -> C -> I -> F -> B -> J -> D -> A Distance: 121.97
Filling 48 missing individuals with random routes.
Generation: 7
Best Route: A -> G -> E -> J -> D -> F -> I -> H -> C -> B -> A Distance: 121.46
Filling 84 missing individual

In [ ]:
population_size = 100
mutation_rate = 0.2
max_generations = 10

for i in range(10):
  best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'A', verbose= False)
  print(f'Run {i+1}')
  print("Best Route Found:", " -> ".join(best_route))
  print(f"Best Distance: {best_distance:.2f}\n")

Run 1
Best Route Found: A -> C -> B -> H -> E -> I -> F -> D -> J -> G -> A
Best Distance: 127.34

Run 2
Best Route Found: A -> G -> H -> E -> J -> D -> F -> C -> I -> B -> A
Best Distance: 121.54

Run 3
Best Route Found: A -> F -> I -> C -> B -> D -> J -> E -> H -> G -> A
Best Distance: 126.61

Run 4
Best Route Found: A -> I -> F -> C -> H -> E -> J -> D -> B -> G -> A
Best Distance: 122.71

Run 5
Best Route Found: A -> C -> H -> E -> G -> J -> D -> B -> I -> F -> A
Best Distance: 118.36

Run 6
Best Route Found: A -> F -> I -> B -> C -> H -> E -> G -> D -> J -> A
Best Distance: 123.47

Run 7
Best Route Found: A -> F -> I -> C -> H -> B -> J -> D -> E -> G -> A
Best Distance: 131.58

Run 8
Best Route Found: A -> F -> C -> H -> B -> I -> E -> J -> D -> G -> A
Best Distance: 128.03

Run 9
Best Route Found: A -> B -> D -> J -> F -> I -> C -> H -> E -> G -> A
Best Distance: 123.40

Run 10
Best Route Found: A -> I -> F -> D -> J -> G -> E -> H -> C -> B -> A
Best Distance: 123.84



In [ ]:
# tuning 1: increase generations
population_size = 100
mutation_rate = 0.2
max_generations = 100

result = []
for i in range(100):
  best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'A', verbose= False)
  result.append( best_distance)

df = pd.DataFrame(result)
df.describe()

,0
count,100.00000
mean,110.22880
std,5.10982
min,101.34000
25%,106.11000
50%,111.80000
75%,114.14750
max,121.43000


In [ ]:
# tuning 1: increase generations
population_size = 100
mutation_rate = 0.2
max_generations = 200

result = []
for i in range(100):
  best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'A', verbose= False)
  result.append( best_distance)

df = pd.DataFrame(result)
df.describe()

,0
count,100.000000
mean,107.504600
std,4.845822
min,101.340000
25%,103.600000
50%,108.070000
75%,112.230000
max,119.050000


In [ ]:
# tuning 2: increase population
population_size = 200
mutation_rate = 0.2
max_generations = 200

result = []
for i in range(100):
  best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'A', verbose= False)
  result.append( best_distance)

df = pd.DataFrame(result)
df.describe()

,0
count,100.000000
mean,105.593400
std,4.094404
min,101.340000
25%,101.340000
50%,103.600000
75%,108.070000
max,114.860000


In [ ]:
# tuning 2: increase population
population_size = 400
mutation_rate = 0.2
max_generations = 200

result = []
for i in range(100):
  best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'A', verbose= False)
  result.append( best_distance)

df = pd.DataFrame(result)
df.describe()

,0
count,100.000000
mean,104.037600
std,3.036434
min,101.340000
25%,101.340000
50%,103.600000
75%,106.110000
max,112.230000


In [ ]:
# tuning 2: increase mutation rate
population_size = 400
mutation_rate = 0.3
max_generations = 200

result = []
for i in range(100):
  best_route, best_distance = genetic_algorithm(cities, distances, population_size, mutation_rate, max_generations,start_city = 'A', verbose= False)
  result.append( best_distance)

df = pd.DataFrame(result)
df.describe()

,0
count,100.000000
mean,104.013400
std,2.940613
min,101.340000
25%,101.340000
50%,103.600000
75%,106.110000
max,111.800000


## PyGAD Implementation

In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

# Distance matrix as a NumPy array
distance_matrix = np.array([
    [0, 18.56, 18.27, 18.89, 27.02, 18.72, 5.03, 23.32, 17.33, 17.83],
    [18.56, 0, 21.16, 22.32, 31.64, 26.57, 21.11, 12.62, 9.85, 22.46],
    [18.27, 21.16, 0, 33.11, 20.42, 18.94, 25.32, 5.46, 16.88, 24.9],
    [18.89, 22.32, 33.11, 0, 28.73, 17.59, 23.77, 22.48, 27.82, 8.87],
    [27.02, 31.64, 20.42, 28.73, 0, 25.68, 5, 5, 13.93, 10.84],
    [18.72, 26.57, 18.94, 17.59, 25.68, 0, 28.76, 26.64, 7.81, 28.47],
    [5.03, 21.11, 25.32, 23.77, 5, 28.76, 0, 9.98, 19.14, 17.06],
    [23.32, 12.62, 5.46, 22.48, 5, 26.64, 9.98, 0, 21.14, 28.19],
    [17.33, 9.85, 16.88, 27.82, 13.93, 7.81, 19.14, 21.14, 0, 26.38],
    [17.83, 22.46, 24.9, 8.87, 10.84, 28.47, 17.06, 28.19, 26.38, 0]
])

# Number of cities
num_cities = len(distance_matrix)

# Create the fitness function and individual class
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

# Initialize the toolbox
toolbox = base.Toolbox()

# Attribute generator: random permutation of cities
toolbox.register("indices", random.sample, range(num_cities), num_cities)

# Structure initializers
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Evaluation function
def evalTSP(individual):
    distance = 0
    for i in range(num_cities):
        distance += distance_matrix[individual[i - 1]][individual[i]]
    return distance,

toolbox.register("mate", tools.cxOrdered)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evalTSP)

def main():
    random.seed(42)
    pop = toolbox.population(n=400)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("min", np.min)
    stats.register("avg", np.mean)

    algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2, ngen=200, stats=stats, halloffame=hof, verbose=True)

    return pop, stats, hof

if __name__ == "__main__":
    pop, stats, hof = main()
    best_route = hof[0]
    print("Best route found:", best_route)
    print("Distance of the best route:", evalTSP(best_route)[0])

/usr/local/lib/python3.10/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMin' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.10/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


gen	nevals	min   	avg    
0  	400   	130.53	194.986
1  	323   	128.41	181.152
2  	302   	114.22	172.739
3  	298   	101.34	165.212
4  	305   	118.69	161.858
5  	295   	114.4 	156.385
6  	289   	103.6 	152.145
7  	307   	103.6 	149.49 
8  	311   	103.6 	146.725
9  	302   	103.6 	146.165
10 	306   	101.34	144.026
11 	305   	101.34	142.975
12 	293   	101.34	139.157
13 	312   	101.34	137.046
14 	304   	101.34	133.648
15 	284   	101.34	129.37 
16 	297   	101.34	127.168
17 	322   	101.34	124.988
18 	297   	101.34	121.15 
19 	310   	101.34	119.657
20 	304   	101.34	116.706
21 	303   	101.34	120.905
22 	299   	101.34	119.232
23 	311   	101.34	117.013
24 	301   	101.34	116.332
25 	318   	101.34	116.166
26 	320   	101.34	115.459
27 	320   	101.34	113.801
28 	306   	101.34	113.485
29 	315   	101.34	110.973
30 	317   	101.34	108.094
31 	312   	101.34	105.512
32 	308   	101.34	104.952
33 	274   	101.34	104.662
34 	300   	101.34	104.752
35 	309   	101.34	106.661
36 	305   	101.34	104.433
37 	298   	1

In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

# Distance matrix as a NumPy array
distance_matrix = np.array([
    [0, 18.56, 18.27, 18.89, 27.02, 18.72, 5.03, 23.32, 17.33, 17.83],
    [18.56, 0, 21.16, 22.32, 31.64, 26.57, 21.11, 12.62, 9.85, 22.46],
    [18.27, 21.16, 0, 33.11, 20.42, 18.94, 25.32, 5.46, 16.88, 24.9],
    [18.89, 22.32, 33.11, 0, 28.73, 17.59, 23.77, 22.48, 27.82, 8.87],
    [27.02, 31.64, 20.42, 28.73, 0, 25.68, 5, 5, 13.93, 10.84],
    [18.72, 26.57, 18.94, 17.59, 25.68, 0, 28.76, 26.64, 7.81, 28.47],
    [5.03, 21.11, 25.32, 23.77, 5, 28.76, 0, 9.98, 19.14, 17.06],
    [23.32, 12.62, 5.46, 22.48, 5, 26.64, 9.98, 0, 21.14, 28.19],
    [17.33, 9.85, 16.88, 27.82, 13.93, 7.81, 19.14, 21.14, 0, 26.38],
    [17.83, 22.46, 24.9, 8.87, 10.84, 28.47, 17.06, 28.19, 26.38, 0]
])

# Number of cities
num_cities = len(distance_matrix)

# Specify the starting city (e.g., 'A' is index 0)
start_idx = 0

# Create the fitness function and individual class
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

# Initialize the toolbox
toolbox = base.Toolbox()

# Attribute generator: random permutation of cities excluding the starting city
def random_permutation():
    return random.sample([i for i in range(num_cities) if i != start_idx], num_cities - 1)

toolbox.register("indices", random_permutation)

# Structure initializers
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Evaluation function
def evalTSP(individual):
    # Add the starting city at the beginning and end of the route
    route = [start_idx] + individual + [start_idx]
    distance = 0
    for i in range(len(route) - 1):
        distance += distance_matrix[route[i]][route[i + 1]]
    return distance,

# Custom crossover function to handle the fixed starting city
def fixed_start_crossover(ind1, ind2):
    size = len(ind1)
    start, end = sorted(random.sample(range(size), 2))
    child1 = [-1] * size
    child2 = [-1] * size

    # Copy crossover segment from first parent
    child1[start:end] = ind1[start:end]
    child2[start:end] = ind2[start:end]

    # Fill the rest from the other parent
    remaining1 = [gene for gene in ind2 if gene not in child1]
    remaining2 = [gene for gene in ind1 if gene not in child2]

    pointer1, pointer2 = 0, 0
    for i in range(size):
        if child1[i] == -1:
            child1[i] = remaining1[pointer1]
            pointer1 += 1
        if child2[i] == -1:
            child2[i] = remaining2[pointer2]
            pointer2 += 1

    return creator.Individual(child1), creator.Individual(child2)

toolbox.register("mate", fixed_start_crossover)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evalTSP)

def main():
    # Seed for reproducibility
    random.seed(42)

    # Initialize the population
    pop = toolbox.population(n=400)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("min", np.min)
    stats.register("avg", np.mean)

    # Run the genetic algorithm
    algorithms.eaSimple(
        pop,
        toolbox,
        cxpb=0.7,
        mutpb=0.2,
        ngen=200,
        stats=stats,
        halloffame=hof,
        verbose=True
    )

    # Return the population, statistics, and the Hall of Fame
    return pop, stats, hof

if __name__ == "__main__":
    pop, stats, hof = main()
    best_route = hof[0]

    print("Best route found (indices):", [start_idx] + best_route + [start_idx])
    print("Distance of the best route:", evalTSP(best_route)[0])

gen	nevals	min   	avg    
0  	400   	136.36	195.285
1  	300   	128.82	181.651
2  	295   	116.58	173.183
3  	312   	116.58	167.364
4  	310   	112.84	160.358
5  	300   	101.34	153.099
6  	299   	101.34	150.704
7  	299   	101.34	142.931
8  	330   	101.34	137.628
9  	291   	101.34	130.814
10 	305   	101.34	125.423
11 	314   	101.34	120.041
12 	300   	101.34	116.665
13 	302   	101.34	112.402
14 	317   	101.34	106.849
15 	295   	101.34	104.369
16 	318   	101.34	103.986
17 	321   	101.34	105.537
18 	306   	101.34	104.087
19 	303   	101.34	104.754
20 	286   	101.34	104.123
21 	300   	101.34	104.418
22 	310   	101.34	105.169
23 	303   	101.34	104.272
24 	296   	101.34	105.34 
25 	298   	101.34	105.444
26 	312   	101.34	105.442
27 	314   	101.34	105.002
28 	313   	101.34	104.672
29 	314   	101.34	105.53 
30 	288   	101.34	105.265
31 	316   	101.34	104.79 
32 	301   	101.34	104.337
33 	315   	101.34	104.444
34 	303   	101.34	104.309
35 	299   	101.34	104.717
36 	312   	101.34	104.807
37 	304   	1

In [ ]:
import pandas as pd

def run_algorithm():
    random.seed(42)
    pop = toolbox.population(n=400)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("min", np.min)
    stats.register("avg", np.mean)

    algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2, ngen=200, stats=stats, halloffame=hof, verbose=False)
    return evalTSP(hof[0])[0]  # Return the best distance

results = [run_algorithm() for _ in range(100)]
df = pd.DataFrame(results, columns=["Best Distance"])
pd.options.display.float_format = '{:.2f}'.format
df.describe()

df = pd.DataFrame(results, columns=["Best Distance"])
pd.options.display.float_format = '{:.2f}'.format
df.describe()

,Best Distance
count,100.00
mean,101.34
std,0.00
min,101.34
25%,101.34
50%,101.34
75%,101.34
max,101.34
